# Notebook 00 — เตรียมข้อมูล Sounding เชียงใหม่จาก GitHub

**รายวิชา:** บรรยากาศและคุณภาพอากาศ / Atmosphere and Air Quality  
**ระดับ:** นิสิตสิ่งแวดล้อม ปี 3–4  
**สถานี:** Chiang Mai, WMO **48327**  
**ช่วงข้อมูล:** 1 มีนาคม–30 เมษายน 2024  
**เวลาที่ใช้:** **00 UTC = 07:00 ICT**  
**จำนวนที่คาดหวัง:** **61 soundings**

Repository:

`https://github.com/nattaponm/Teaching_Sounding_AirQuality`

## วัตถุประสงค์

Notebook นี้เป็นไฟล์เตรียมข้อมูลที่นิสิตต้องรันก่อน Notebook 01–04 เพื่อให้ทุกคนใช้ dataset เดียวกัน โดยจะทำงานตามลำดับ

**GitHub → Download → Metadata → Validation → 00 UTC dataset → Google Drive**

Notebook นี้ยังไม่คำนวณ CAPE, CIN, inversion หรือ PBL height เพราะการวิเคราะห์เชิงกายภาพจะเริ่มในบทถัดไป

## 0.1 ทำไมต้องใช้ 00 UTC

ประเทศไทยใช้เวลา ICT = UTC+7 ดังนั้น

\[
00\ UTC = 07{:}00\ ICT
\]

ข้อมูลชุดนี้จึงเป็น **morning atmospheric profile** ไม่ใช่ค่าตัวแทนสูงสุดของทั้งวัน

เมื่อนำไปวิเคราะห์ boundary layer หรือ atmospheric stability ในบทต่อไป ต้องตีความผลในบริบทของ **07:00 ICT sounding**

In [1]:
# CELL 1 — Mount Google Drive
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

Mounted at /content/drive


In [2]:
# CELL 2 — Course configuration
from pathlib import Path
import hashlib
import requests
import numpy as np
import pandas as pd

GITHUB_USER = "nattaponm"
GITHUB_REPO = "Teaching_Sounding_AirQuality"
GITHUB_BRANCH = "main"

RAW_BASE = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}"
)

FILES = {
    "dataset": "chiangmai_48327_sounding_20240301_20240430.csv.gz",
    "manifest": "launch_manifest_48327_20240301_20240430.csv",
    "metadata": "station_metadata_48327.csv",
    "report": "download_report_48327_20240301_20240430.txt",
    "readme": "README_DATASET.md",
}

EXPECTED_STATION = "48327"
EXPECTED_HOUR_UTC = 0
EXPECTED_START = "2024-03-01"
EXPECTED_END = "2024-04-30"
EXPECTED_N = 61

BASE_DIR = Path(
    "/content/drive/MyDrive/Teaching_Sounding_AirQuality"
)

SOURCE_DIR = BASE_DIR / "00_source_github"
DATA_DIR = BASE_DIR / "01_data"
META_DIR = BASE_DIR / "02_metadata"
OUTPUT_DIR = BASE_DIR / "03_output"
FIG_DIR = BASE_DIR / "04_figures"

for folder in [
    SOURCE_DIR,
    DATA_DIR,
    META_DIR,
    OUTPUT_DIR,
    FIG_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

TEACHING_DATA = (
    DATA_DIR
    / "chiangmai_48327_00UTC_20240301_20240430.csv.gz"
)

print("Repository:")
print(f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}")

print("\nWorking directory:")
print(BASE_DIR)

Repository:
https://github.com/nattaponm/Teaching_Sounding_AirQuality

Working directory:
/content/drive/MyDrive/Teaching_Sounding_AirQuality


## 0.2 ดาวน์โหลดไฟล์จาก GitHub

ไฟล์หลักที่ใช้มี 5 ไฟล์:

1. combined sounding dataset
2. launch manifest
3. station metadata
4. download report
5. dataset README

ฟังก์ชันด้านล่างจะไม่ดาวน์โหลดซ้ำหากมีไฟล์อยู่แล้ว และจะคำนวณ SHA-256 checksum เพื่อช่วยตรวจสอบความคงที่ของ dataset

In [3]:
# CELL 3 — Download helper functions
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


def download_file(url, destination, force=False):

    destination = Path(destination)

    if (
        destination.exists()
        and destination.stat().st_size > 0
        and not force
    ):
        return {
            "file": destination.name,
            "status": "cached",
            "size_MB": destination.stat().st_size / (1024 ** 2),
            "sha256": sha256_file(destination)
        }

    response = requests.get(
        url,
        timeout=60
    )

    response.raise_for_status()

    if len(response.content) == 0:
        raise RuntimeError(
            f"Empty file returned: {destination.name}"
        )

    temporary = Path(
        str(destination) + ".part"
    )

    temporary.write_bytes(
        response.content
    )

    temporary.replace(
        destination
    )

    return {
        "file": destination.name,
        "status": "downloaded",
        "size_MB": destination.stat().st_size / (1024 ** 2),
        "sha256": sha256_file(destination)
    }

In [4]:
# CELL 4 — Download all teaching source files
records = []

for key, filename in FILES.items():

    url = f"{RAW_BASE}/{filename}"
    destination = SOURCE_DIR / filename

    try:
        result = download_file(
            url,
            destination,
            force=False
        )

        result["key"] = key
        result["url"] = url
        records.append(
            result
        )

    except Exception as exc:

        records.append({
            "key": key,
            "file": filename,
            "status": "ERROR",
            "size_MB": np.nan,
            "sha256": None,
            "url": url,
            "error": str(exc)
        })


download_summary = pd.DataFrame(
    records
)

print("=== GITHUB DOWNLOAD SUMMARY ===")

display(
    download_summary[
        [
            "key",
            "file",
            "status",
            "size_MB",
            "sha256"
        ]
    ]
)

if (
    download_summary["status"]
    == "ERROR"
).any():

    raise RuntimeError(
        "At least one required file could not be downloaded. "
        "Inspect the table above before continuing."
    )

print("\nAll required files are available.")

=== GITHUB DOWNLOAD SUMMARY ===


,key,file,status,size_MB,sha256
0,dataset,chiangmai_48327_sounding_20240301_20240430.csv.gz,downloaded,0.113939,09a6adc855c64af30ad9ae7704ddf2064bb3f8024ab8e7...
1,manifest,launch_manifest_48327_20240301_20240430.csv,downloaded,0.070498,61d1357b249f7fc76c8745b8434a1ea424fb6e219a1294...
2,metadata,station_metadata_48327.csv,downloaded,0.000428,75a958df10a95209e8dafbbf29de1422e4ec28024f7742...
3,report,download_report_48327_20240301_20240430.txt,downloaded,0.000762,923caa33bc7c9a1bacdb7ed10f760cc50d3a52df62b6fb...
4,readme,README_DATASET.md,downloaded,0.001157,a3fbb19bcbcb04fc24ecd7d16a379a890e5e2f86b7f258...



All required files are available.


## 0.3 ตรวจ Metadata และ Data Availability

ก่อนเปิด sounding profile เราต้องดูว่า dataset มาจากไหนและมีข้อมูลครบหรือไม่

แนวคิดนี้เรียกว่า **data provenance**

สำหรับชุดการสอนนี้เราทราบจากการเตรียมข้อมูลว่า 00 UTC มีข้อมูลครบ 61 วัน ส่วน Notebook นี้จะตรวจซ้ำจากไฟล์ manifest ที่เก็บไว้ใน repository

In [5]:
# CELL 5 — Read metadata, manifest and report
metadata = pd.read_csv(
    SOURCE_DIR / FILES["metadata"]
)

manifest = pd.read_csv(
    SOURCE_DIR / FILES["manifest"]
)

print("=== STATION METADATA ===")
display(metadata)

print("\n=== MANIFEST — FIRST 10 RECORDS ===")
display(
    manifest.head(10)
)

manifest["success"] = (
    manifest["status"]
    .astype(str)
    .isin(["downloaded", "cached"])
)

availability = (
    manifest
    .groupby("hour_utc")
    .agg(
        requested=("status", "size"),
        success=("success", "sum")
    )
    .reset_index()
)

availability["availability_pct"] = (
    100
    * availability["success"]
    / availability["requested"]
)

print("\n=== DATA AVAILABILITY ===")
display(
    availability
)

print("\n=== ORIGINAL DOWNLOAD REPORT ===")

report_text = (
    SOURCE_DIR
    / FILES["report"]
).read_text(
    encoding="utf-8"
)

print(
    report_text
)

=== STATION METADATA ===


,station_id,station_name_expected,station_name_source,latitude,longitude,elevation_m,study_start,study_end,hours_utc_checked,timezone_local,source_name,source_endpoint,prepared_by_notebook
0,48327,Chiang Mai,"CHIANG MAI, THAILAND",18.78,98.98,NaN,2024-03-01,2024-04-30,"00,06,12",Asia/Bangkok (UTC+7),University of Wyoming Atmospheric Science Radi...,https://weather.uwyo.edu/wsgi/sounding,00_download_ChiangMai_48327_Sounding_MarApr202...



=== MANIFEST — FIRST 10 RECORDS ===


,station_id,requested_datetime_utc,datetime_ict,date_utc,hour_utc,hour_ict,status,message,n_levels,surface_pressure_hPa,top_pressure_hPa,max_height_m,pressure_monotonic_decreasing,temperature_valid_n,dewpoint_valid_n,wind_valid_n,wind_speed_source_unit,raw_text_file,profile_csv_file,source_endpoint
0,48327,2024-03-01 00:00:00,2024-03-01 07:00:00,2024-03-01,0,7,cached,existing profile CSV reused,46.0,1000.0,749.0,2812.0,True,46.0,46.0,1.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
1,48327,2024-03-01 06:00:00,2024-03-01 13:00:00,2024-03-01,6,13,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
2,48327,2024-03-01 12:00:00,2024-03-01 19:00:00,2024-03-01,12,19,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
3,48327,2024-03-02 00:00:00,2024-03-02 07:00:00,2024-03-02,0,7,downloaded,downloaded and parsed successfully,53.0,1000.0,700.0,2850.0,True,53.0,53.0,53.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
4,48327,2024-03-02 06:00:00,2024-03-02 13:00:00,2024-03-02,6,13,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
5,48327,2024-03-02 12:00:00,2024-03-02 19:00:00,2024-03-02,12,19,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
6,48327,2024-03-03 00:00:00,2024-03-03 07:00:00,2024-03-03,0,7,downloaded,downloaded and parsed successfully,63.0,1000.0,700.0,2850.0,True,63.0,63.0,63.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
7,48327,2024-03-03 06:00:00,2024-03-03 13:00:00,2024-03-03,6,13,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
8,48327,2024-03-03 12:00:00,2024-03-03 19:00:00,2024-03-03,12,19,error,HTTP 404,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding
9,48327,2024-03-04 00:00:00,2024-03-04 07:00:00,2024-03-04,0,7,downloaded,downloaded and parsed successfully,56.0,1000.0,700.0,3450.0,True,56.0,56.0,56.0,m/s,/content/drive/MyDrive/Teaching_AirQuality/Chi...,/content/drive/MyDrive/Teaching_AirQuality/Chi...,https://weather.uwyo.edu/wsgi/sounding



=== DATA AVAILABILITY ===


,hour_utc,requested,success,availability_pct
0,0,61,61,100.0
1,6,61,0,0.0
2,12,61,0,0.0



=== ORIGINAL DOWNLOAD REPORT ===
Chiang Mai radiosonde download report
WMO station ID: 48327
Expected station name: Chiang Mai
Study period: 2024-03-01 to 2024-04-30
UTC hours checked: [0, 6, 12]
Source endpoint: https://weather.uwyo.edu/wsgi/sounding

Availability summary:
 hour_utc  hour_ict  requested  success  no_data  error  availability_pct
        0         7         61       61        0      0             100.0
        6        13         61        0        0     61               0.0
       12        19         61        0        0     61               0.0

Status counts:
status    cached  downloaded  error  All
hour_utc                                
0              1          60      0   61
6              0           0     61   61
12             0           0     61   61
All            1          60    122  183


## 0.4 อ่าน Combined Sounding Dataset

หนึ่ง radiosonde sounding ประกอบด้วยข้อมูลหลายระดับในแนวดิ่ง ดังนั้นหนึ่งวันจะมีหลาย rows

ตัวแปรสำคัญที่ควรพบ ได้แก่

- pressure
- height
- temperature
- dew point
- relative humidity
- mixing ratio
- wind direction
- wind speed
- potential temperature
- equivalent potential temperature

จำนวน sounding จึงต้องนับจาก `launch_datetime_utc` ที่ไม่ซ้ำกัน ไม่ใช่นับจำนวน rows

In [6]:
# CELL 6 — Read and standardize the combined dataset
source_data = (
    SOURCE_DIR
    / FILES["dataset"]
)

df = pd.read_csv(
    source_data,
    compression="gzip"
)

df["station_id"] = (
    df["station_id"]
    .astype(str)
    .str.replace(
        r"\.0$",
        "",
        regex=True
    )
)

df["launch_datetime_utc"] = pd.to_datetime(
    df["launch_datetime_utc"],
    errors="coerce"
)

df["launch_datetime_ict"] = pd.to_datetime(
    df["launch_datetime_ict"],
    errors="coerce"
)

df["hour_utc"] = pd.to_numeric(
    df["hour_utc"],
    errors="coerce"
)

print("Dataset shape:")
print(df.shape)

print("\nColumns:")
for i, col in enumerate(
    df.columns,
    start=1
):
    print(
        f"{i:02d}. {col}"
    )

print("\nFirst 12 rows:")
display(
    df.head(12)
)

Dataset shape:
(4690, 20)

Columns:
01. station_id
02. launch_datetime_utc
03. launch_datetime_ict
04. hour_utc
05. hour_ict
06. pressure_hPa
07. height_m
08. temperature_C
09. dewpoint_C
10. relative_humidity_pct
11. mixing_ratio_gkg
12. wind_direction_deg
13. wind_speed_source
14. wind_speed_source_unit
15. wind_speed_ms
16. theta_K
17. theta_e_K
18. theta_v_K
19. source_file
20. source_archive

First 12 rows:


,station_id,launch_datetime_utc,launch_datetime_ict,hour_utc,hour_ict,pressure_hPa,height_m,temperature_C,dewpoint_C,relative_humidity_pct,mixing_ratio_gkg,wind_direction_deg,wind_speed_source,wind_speed_source_unit,wind_speed_ms,theta_K,theta_e_K,theta_v_K,source_file,source_archive
0,48327,2024-03-01,2024-03-01 07:00:00,0,7,1000.0,314.0,20.8,19.8,94.0,14.68,0.0,0.0,m/s,0.0,293.9,336.0,296.5,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
1,48327,2024-03-01,2024-03-01 07:00:00,0,7,968.0,597.0,22.4,14.4,61.0,10.71,NaN,NaN,m/s,NaN,298.3,329.7,300.2,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
2,48327,2024-03-01,2024-03-01 07:00:00,0,7,962.0,651.0,23.0,15.0,61.0,11.21,NaN,NaN,m/s,NaN,299.4,332.5,301.5,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
3,48327,2024-03-01,2024-03-01 07:00:00,0,7,957.0,696.0,23.6,14.6,57.0,10.98,NaN,NaN,m/s,NaN,300.5,333.0,302.5,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
4,48327,2024-03-01,2024-03-01 07:00:00,0,7,951.0,751.0,24.0,14.0,54.0,10.62,NaN,NaN,m/s,NaN,301.4,333.0,303.4,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
5,48327,2024-03-01,2024-03-01 07:00:00,0,7,946.0,798.0,24.4,13.4,50.0,10.26,NaN,NaN,m/s,NaN,302.3,333.0,304.2,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
6,48327,2024-03-01,2024-03-01 07:00:00,0,7,940.0,853.0,24.6,12.6,47.0,9.79,NaN,NaN,m/s,NaN,303.1,332.5,304.8,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
7,48327,2024-03-01,2024-03-01 07:00:00,0,7,935.0,900.0,25.0,11.0,41.0,8.85,NaN,NaN,m/s,NaN,303.9,330.7,305.6,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
8,48327,2024-03-01,2024-03-01 07:00:00,0,7,930.0,947.0,25.0,10.0,39.0,8.31,NaN,NaN,m/s,NaN,304.4,329.6,305.9,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...
9,48327,2024-03-01,2024-03-01 07:00:00,0,7,925.0,994.0,25.2,10.2,39.0,8.47,NaN,NaN,m/s,NaN,305.1,330.8,306.6,sounding_48327_2024-03-01_00Z.csv,University of Wyoming Atmospheric Science Radi...


In [7]:
# CELL 7 — Select 00 UTC teaching data
df_00 = (
    df[
        (df["station_id"] == EXPECTED_STATION)
        & (df["hour_utc"] == EXPECTED_HOUR_UTC)
    ]
    .copy()
)

df_00 = (
    df_00
    .sort_values(
        [
            "launch_datetime_utc",
            "pressure_hPa"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)

print("Original rows:")
print(
    len(df)
)

print("\n00 UTC rows:")
print(
    len(df_00)
)

print("\nUnique UTC hours in teaching dataset:")
print(
    sorted(
        df_00["hour_utc"]
        .dropna()
        .unique()
        .tolist()
    )
)

Original rows:
4690

00 UTC rows:
4690

Unique UTC hours in teaching dataset:
[0]


## 0.5 Core Dataset Validation

ก่อนใช้ข้อมูลใน Notebook 01–04 ต้องตรวจว่า:

- Station = 48327
- UTC hour = 00
- First date = 2024-03-01
- Last date = 2024-04-30
- March = 31 soundings
- April = 30 soundings
- Total = 61 soundings

หากรายการใดไม่ผ่าน ให้ตรวจ dataset ก่อนเริ่มวิเคราะห์

In [8]:
# CELL 8 — Validate the course dataset
launches = (
    df_00[
        "launch_datetime_utc"
    ]
    .dropna()
    .drop_duplicates()
    .sort_values()
)

first_date = (
    launches.min()
    .strftime("%Y-%m-%d")
)

last_date = (
    launches.max()
    .strftime("%Y-%m-%d")
)

n_total = len(
    launches
)

month_counts = (
    launches
    .dt.month
    .value_counts()
)

n_march = int(
    month_counts.get(
        3,
        0
    )
)

n_april = int(
    month_counts.get(
        4,
        0
    )
)

stations = sorted(
    df_00["station_id"]
    .dropna()
    .unique()
    .tolist()
)

hours = sorted(
    df_00["hour_utc"]
    .dropna()
    .unique()
    .tolist()
)

required_columns = [
    "pressure_hPa",
    "height_m",
    "temperature_C",
    "dewpoint_C",
    "relative_humidity_pct",
    "mixing_ratio_gkg",
    "wind_direction_deg",
    "wind_speed_ms",
    "theta_K",
    "theta_e_K",
    "theta_v_K"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df_00.columns
]

checks = pd.DataFrame([
    {
        "check": "Station ID",
        "observed": str(stations),
        "expected": "['48327']",
        "pass": stations == ["48327"]
    },
    {
        "check": "UTC hour",
        "observed": str(hours),
        "expected": "[0]",
        "pass": hours == [0]
    },
    {
        "check": "First date",
        "observed": first_date,
        "expected": EXPECTED_START,
        "pass": first_date == EXPECTED_START
    },
    {
        "check": "Last date",
        "observed": last_date,
        "expected": EXPECTED_END,
        "pass": last_date == EXPECTED_END
    },
    {
        "check": "March soundings",
        "observed": n_march,
        "expected": 31,
        "pass": n_march == 31
    },
    {
        "check": "April soundings",
        "observed": n_april,
        "expected": 30,
        "pass": n_april == 30
    },
    {
        "check": "Total soundings",
        "observed": n_total,
        "expected": EXPECTED_N,
        "pass": n_total == EXPECTED_N
    },
    {
        "check": "Required variables",
        "observed": (
            "all present"
            if not missing_columns
            else str(missing_columns)
        ),
        "expected": "all present",
        "pass": len(missing_columns) == 0
    }
])

checks["status"] = np.where(
    checks["pass"],
    "PASS",
    "CHECK"
)

print("=== CORE DATASET VALIDATION ===")

display(
    checks[
        [
            "check",
            "observed",
            "expected",
            "status"
        ]
    ]
)

if checks["pass"].all():
    print(
        "\nDATASET STATUS: READY"
    )
else:
    raise RuntimeError(
        "Core dataset validation did not fully pass."
    )

=== CORE DATASET VALIDATION ===


,check,observed,expected,status
0,Station ID,['48327'],['48327'],PASS
1,UTC hour,[0],[0],PASS
2,First date,2024-03-01,2024-03-01,PASS
3,Last date,2024-04-30,2024-04-30,PASS
4,March soundings,31,31,PASS
5,April soundings,30,30,PASS
6,Total soundings,61,61,PASS
7,Required variables,all present,all present,PASS



DATASET STATUS: READY


## 0.6 จำนวนระดับและ Structural QC เบื้องต้น

จำนวนระดับของแต่ละ sounding ไม่จำเป็นต้องเท่ากันทุกวัน

เราจะตรวจเพียง structural QC เบื้องต้น:

1. จำนวนระดับต่อ sounding
2. pressure ลดลงเมื่อขึ้นสู่ระดับสูง
3. จำนวนกรณีที่ dew point สูงกว่า temperature มากกว่า 0.2°C

ขั้นตอนนี้เป็น **QC flagging** ไม่ใช่การลบข้อมูล

In [9]:
# CELL 9 — Levels per sounding and simple structural QC
level_summary = (
    df_00
    .groupby(
        "launch_datetime_utc"
    )
    .size()
    .rename(
        "n_levels"
    )
    .reset_index()
)

qc_rows = []

for launch, profile in df_00.groupby(
    "launch_datetime_utc"
):

    profile = profile.sort_values(
        "pressure_hPa",
        ascending=False
    )

    p = pd.to_numeric(
        profile["pressure_hPa"],
        errors="coerce"
    ).dropna()

    monotonic = (
        bool(
            (
                p.diff()
                .dropna()
                <= 0
            ).all()
        )
        if len(p) >= 2
        else False
    )

    t = pd.to_numeric(
        profile["temperature_C"],
        errors="coerce"
    )

    td = pd.to_numeric(
        profile["dewpoint_C"],
        errors="coerce"
    )

    valid = (
        t.notna()
        & td.notna()
    )

    td_gt_t = int(
        (
            td[valid]
            > t[valid] + 0.2
        ).sum()
    )

    qc_rows.append({
        "launch_datetime_utc": launch,
        "n_levels": len(profile),
        "pressure_monotonic_decreasing": monotonic,
        "Td_gt_T_plus_0p2_count": td_gt_t
    })


qc = pd.DataFrame(
    qc_rows
)

print("=== NUMBER OF LEVELS PER SOUNDING ===")
display(
    level_summary[
        "n_levels"
    ]
    .describe()
    .to_frame()
    .T
)

print(
    "\nPressure-order checks passed:",
    int(
        qc[
            "pressure_monotonic_decreasing"
        ].sum()
    ),
    "/",
    len(qc)
)

print(
    "Soundings with Td > T + 0.2°C:",
    int(
        (
            qc[
                "Td_gt_T_plus_0p2_count"
            ]
            > 0
        ).sum()
    )
)

=== NUMBER OF LEVELS PER SOUNDING ===


,count,mean,std,min,25%,50%,75%,max
n_levels,61.0,76.885246,12.244861,46.0,71.0,77.0,84.0,117.0



Pressure-order checks passed: 61 / 61
Soundings with Td > T + 0.2°C: 0


## 0.7 ดูข้อมูล Sounding ตัวอย่าง

ตัวอย่างใช้วันที่ **15 มีนาคม 2024**

ใน Notebook 01 จะเริ่มเรียนความหมายทางกายภาพของตัวแปรแต่ละตัวจาก vertical profile จริง

In [10]:
# CELL 10 — Inspect one sounding
EXAMPLE_DATE = "2024-03-15"

example = df_00[
    df_00[
        "launch_datetime_utc"
    ]
    .dt.strftime(
        "%Y-%m-%d"
    )
    == EXAMPLE_DATE
].copy()

columns_to_show = [
    "launch_datetime_utc",
    "launch_datetime_ict",
    "pressure_hPa",
    "height_m",
    "temperature_C",
    "dewpoint_C",
    "relative_humidity_pct",
    "mixing_ratio_gkg",
    "wind_direction_deg",
    "wind_speed_ms",
    "theta_K",
    "theta_e_K"
]

columns_to_show = [
    col
    for col in columns_to_show
    if col in example.columns
]

print(
    f"Example sounding: {EXAMPLE_DATE}"
)

print(
    "00 UTC = 07:00 ICT"
)

print(
    "Vertical levels:",
    len(example)
)

display(
    example[
        columns_to_show
    ].head(20)
)

Example sounding: 2024-03-15
00 UTC = 07:00 ICT
Vertical levels: 89


,launch_datetime_utc,launch_datetime_ict,pressure_hPa,height_m,temperature_C,dewpoint_C,relative_humidity_pct,mixing_ratio_gkg,wind_direction_deg,wind_speed_ms,theta_K,theta_e_K
984,2024-03-15,2024-03-15 07:00:00,977.0,314.0,21.4,16.6,74.0,12.25,0.0,0.0,296.5,332.1
985,2024-03-15,2024-03-15 07:00:00,976.0,323.0,22.0,14.6,63.0,10.76,10.0,8.2,297.2,328.6
986,2024-03-15,2024-03-15 07:00:00,975.0,332.0,22.6,12.6,53.0,9.44,9.0,7.1,297.9,325.6
987,2024-03-15,2024-03-15 07:00:00,969.0,386.0,23.3,12.1,49.0,9.16,0.0,0.0,299.1,326.2
988,2024-03-15,2024-03-15 07:00:00,955.0,514.0,24.8,10.8,41.0,8.54,337.0,0.2,301.9,327.5
989,2024-03-15,2024-03-15 07:00:00,925.0,796.0,24.6,11.6,44.0,9.31,285.0,0.5,304.5,332.6
990,2024-03-15,2024-03-15 07:00:00,914.0,901.0,24.6,10.6,41.0,8.81,304.0,1.4,305.5,332.3
991,2024-03-15,2024-03-15 07:00:00,868.0,1350.0,22.0,3.8,30.0,5.78,25.0,5.2,307.4,325.4
992,2024-03-15,2024-03-15 07:00:00,850.0,1532.0,21.0,1.0,26.0,4.84,40.0,6.2,308.1,323.4
993,2024-03-15,2024-03-15 07:00:00,838.0,1655.0,20.4,-0.6,24.0,4.37,44.0,6.6,308.8,322.6


## 0.8 สร้าง Analysis-ready Teaching Dataset

Notebook ต่อไปจะอ่านไฟล์นี้โดยตรง:

`MyDrive/Teaching_Sounding_AirQuality/01_data/chiangmai_48327_00UTC_20240301_20240430.csv.gz`

ดังนั้น Notebook 01–04 ไม่ต้องดาวน์โหลดข้อมูลจาก GitHub ซ้ำ

In [11]:
# CELL 11 — Save teaching dataset and course metadata
df_00.to_csv(
    TEACHING_DATA,
    index=False,
    compression="gzip"
)

COURSE_INFO = (
    META_DIR
    / "course_dataset_info.csv"
)

QC_FILE = (
    META_DIR
    / "basic_structural_QC_48327_00UTC.csv"
)

LEVEL_FILE = (
    META_DIR
    / "levels_per_sounding_48327_00UTC.csv"
)

DOWNLOAD_FILE = (
    META_DIR
    / "github_download_summary.csv"
)

course_info = pd.DataFrame([
    {
        "station_id": EXPECTED_STATION,
        "station_name": "Chiang Mai",
        "hour_utc": 0,
        "hour_ict": 7,
        "study_start": EXPECTED_START,
        "study_end": EXPECTED_END,
        "n_soundings": n_total,
        "n_march": n_march,
        "n_april": n_april,
        "teaching_dataset": str(
            TEACHING_DATA
        ),
        "sha256": sha256_file(
            TEACHING_DATA
        ),
        "github_repository": (
            f"https://github.com/"
            f"{GITHUB_USER}/"
            f"{GITHUB_REPO}"
        )
    }
])

course_info.to_csv(
    COURSE_INFO,
    index=False
)

qc.to_csv(
    QC_FILE,
    index=False
)

level_summary.to_csv(
    LEVEL_FILE,
    index=False
)

download_summary.to_csv(
    DOWNLOAD_FILE,
    index=False
)

print("=== TEACHING DATASET READY ===")

print(
    "Station       : 48327"
)

print(
    "Time          : 00 UTC = 07:00 ICT"
)

print(
    f"Period        : {first_date} to {last_date}"
)

print(
    f"March         : {n_march} soundings"
)

print(
    f"April         : {n_april} soundings"
)

print(
    f"Total         : {n_total} soundings"
)

print(
    "\nTeaching dataset:"
)

print(
    TEACHING_DATA
)

print(
    "\nFile size (MB):",
    round(
        TEACHING_DATA.stat().st_size
        / (1024 ** 2),
        3
    )
)

print(
    "\nSHA-256:"
)

print(
    sha256_file(
        TEACHING_DATA
    )
)

=== TEACHING DATASET READY ===
Station       : 48327
Time          : 00 UTC = 07:00 ICT
Period        : 2024-03-01 to 2024-04-30
March         : 31 soundings
April         : 30 soundings
Total         : 61 soundings

Teaching dataset:
/content/drive/MyDrive/Teaching_Sounding_AirQuality/01_data/chiangmai_48327_00UTC_20240301_20240430.csv.gz

File size (MB): 0.113

SHA-256:
31ba17a2b625f59787fb3cd971275ec9eac7e0117b0550530833bc1ad920cc8a


# Final Checkpoint

เมื่อ Cell สุดท้ายแสดง

```text
Station       : 48327
Time          : 00 UTC = 07:00 ICT
March         : 31 soundings
April         : 30 soundings
Total         : 61 soundings
```

ถือว่า dataset พร้อมสำหรับการเรียน

## Notebook ต่อไป

### `01_understanding_sounding_data_and_QC.ipynb`

จะเริ่มจากการเรียน:

- Pressure และ geopotential height
- Temperature และ dew point
- Relative humidity และ mixing ratio
- Wind speed/direction
- Potential temperature
- การแสดง vertical profiles
- Missing data และ scientific QC
- การเปรียบเทียบ sounding หลายวัน

ตั้งแต่ Notebook 01 เป็นต้นไป เราจะใช้ไฟล์ analysis-ready ที่สร้างใน Google Drive โดยตรง